In [246]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_market_calendars as mcal #for holiday calendars

In [247]:
#Data inputs
df = pd.read_csv('sx7e_prices2.csv', parse_dates=['Date'], index_col=['Date']) #Very simply reads all the data from the CSV for SX7E
#df2 = pd.read_csv('bckttuse_prices.csv', parse_dates=['Date'], index_col=['Date']) 
#df = df.join(df2['Price'], rsuffix="2", how='outer') #Joins the price of SPX with the SX7E date

volStrike1 = 21.45
varAmount1 = 1864.8
lowerBound = 119.35
upperBound = 9999


In [248]:
#nyse = mcal.get_calendar('NYSE') #grabs market holiday calendars
eurex = mcal.get_calendar('EUREX')
start_date = df.index.min()
end_date = df.index.max()
#nyse_days = nyse.valid_days(start_date=start_date, end_date=end_date)
eurex_days = eurex.valid_days(start_date=start_date, end_date=end_date)
#common_days = nyse_days.intersection(eurex_days)
#N_correct = len(common_days)-1 #I subtract 1 to get rid of the first day, which does not count towards the calculation
#N_correct = len(eurex_days)-1
#removing common bad days which cause an issue for eurex:
eurex_days_clean = eurex_days[
    ~((eurex_days.month == 12) & (eurex_days.day.isin([24, 31])))]
N_correct=len(eurex_days_clean)-1

eurex_days_clean = pd.DatetimeIndex(eurex_days_clean).tz_localize(None)

# now alignment works
df = df.loc[df.index.intersection(eurex_days_clean)]

In [249]:
#Calculates log returns using shift and log
df['logReturns1'] = np.log(df['Price']/df['Price'].shift(1))

#Creates corridor condition and then converts the True/False to integer i.e. 1/0 so that it can be used operationally
df['conditionFlag']=((df['Price']<=upperBound) & (df['Price']>=lowerBound) & (df['Price'].shift(1)<=upperBound) & (df['Price'].shift(1)>=lowerBound))
df['conditionFlag']=df['conditionFlag'].astype(int)

#Multiplies log return by barrier condition
df['condLogReturns1'] = df['logReturns1']*df['conditionFlag']

#Squares log return
df['squaredLogReturns1'] = df['condLogReturns1']**2

#Sum returns
sumSquaredReturns1 = df['squaredLogReturns1'].sum()

#Days in range
daysInRange = df['conditionFlag'].sum()

#Realized volatility
FRV = 100*np.sqrt((252*sumSquaredReturns1)/daysInRange)

#Variance adjustment factor
varAdjustment = daysInRange/N_correct

#Equity amount/payoff
payoff = varAmount1*varAdjustment*((FRV**2)-(volStrike1**2))
print(payoff)
print(N_correct)

143779.5329310918
402


In [ ]:
#PERFECTLY ALIGN WITH THE SX7E TEST UPVAR. NOW WANT TO TRY SOME OTHER ONES THAT WE GOT FROM CPs TO CHECK IF ALSO IN LINE.
#THEN ONCE COMFORTABLE, TRY TO CREATE THE Xcorr REALIZED PRICER, WILL BE A LOT OF THE SAME REPETITIVE STUFF, SO TRY REMEMBER